# COMxL-Py — classical optimisation with the working shown

A pure-Python port of the COMxL Excel/VBA optimisation workbook.

The original was a set of VBA user forms driving worksheet formulas: you typed
an objective into a cell, pointed a form at that cell, and the macro wrote its
iterations back onto the sheet. This package keeps the algorithms and the
step-by-step output, but the objective is now an ordinary Python function.

Two things changed in the port:

1. **Objectives are functions, not cells.** You write `lambda x: x**2 - 4*x + 3`
   instead of typing a formula into `B4` and pointing a RefEdit at it.
2. **Steps are still printed.** Every solver takes `show_steps=True` (the
   default) and prints the same iteration table the VBA wrote to the sheet.
   Pass `show_steps=False` for the final answer alone — the table is kept
   either way, as a pandas DataFrame on `result.steps`.

Only numpy, pandas, matplotlib and seaborn are used. No optimisation library:
Gauss elimination, Bairstow root-finding and the simplex pivots are all written
out, since providing them is the point.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from comxl import (gradientsearch, directsearch, constrained,
                   linprog, calculus, matrixops, plotting)

plotting.set_style()
pd.set_option('display.float_format', lambda v: f'{v:.6g}')

print('COMxL-Py ready')

## 1. Three ways to write an objective

The adapter inspects what you pass and works out which form you meant, so you
can use whichever reads best for the problem. All three of these are the same
function.

In [ ]:
f_array   = lambda x: x[0]**2 + 3*x[0]*x[1] + 2*x[1]**2   # one array argument
f_scalars = lambda x1, x2: x1**2 + 3*x1*x2 + 2*x2**2      # separate scalars
f_string  = "X_1^2 + 3*X_1*X_2 + 2*X_2^2"                 # worksheet-style

point = [1.0, 2.0]
for label, f in [('array', f_array), ('scalars', f_scalars), ('string', f_string)]:
    g = calculus.gradient(f, point, show_steps=False).extras['gradient']
    print(f'{label:8s} gradient at {point} = {np.round(g, 6)}')

The string form exists so that formulas lifted straight out of the old
worksheets still run — `^` means power and the Excel function names are
accepted. For new work, a plain Python callable is more capable.

## 2. One-dimensional search

Eight methods, ported from UserForm7 through UserForm13 plus `Minimize1D`.
Take a function whose minimum we know exactly:

$$f(x) = x^4 - 3x^3 + 2, \qquad x^* = 2.25,\quad f(x^*) = -6.54296875$$

In [ ]:
quartic = lambda x: x**4 - 3*x**3 + 2

ax = plotting.plot_function(quartic, -1, 3.5, title='$f(x) = x^4 - 3x^3 + 2$')
ax.scatter([2.25], [quartic(2.25)], color='crimson', s=90, zorder=6,
           label='true minimum')
ax.legend()
plt.show()

### 2.1 Golden section search

The interval shrinks by the factor $\tau = 0.381967$ each pass, and one
interior point is reused, so each iteration costs a single new function
evaluation. This is the default output — the table is what the VBA form wrote
onto the worksheet.

In [ ]:
r_golden = gradientsearch.goldensection(quartic, 0, 4, tol=1e-6)

Note the termination test: the VBA stopped on
$|f(\alpha_1) - f(\alpha_2)| < \text{tol}$, a *function-value* difference,
rather than the more usual interval width. That is preserved here, so numbers
match the spreadsheet.

### 2.2 The other bracketing methods

Same problem, `show_steps=False` so we can compare them side by side.

In [ ]:
runs = {
    'Golden section':   gradientsearch.goldensection(quartic, 0, 4, tol=1e-8, max_iter=300, show_steps=False),
    'Interval halving': gradientsearch.intervalhalving(quartic, 0, 4, tol=1e-8, max_iter=500, show_steps=False),
    'Dichotomous':      gradientsearch.dichotomous(quartic, 0, 4, tol=1e-6, delta=1e-8, show_steps=False),
    'Fibonacci':        gradientsearch.fibonacci(quartic, 0, 4, n=30, show_steps=False),
    'Exhaustive':       gradientsearch.exhaustive(quartic, 0, 4, n=500, show_steps=False),
    'Divide & conquer': gradientsearch.divide_conquer(quartic, 0, 4, n=12, show_steps=False),
    'Secant':           gradientsearch.secant(quartic, 1, 4, tol=1e-8, show_steps=False),
    'Newton-Raphson':   gradientsearch.newtonraphson(quartic, 3.0, tol=1e-10, max_iter=50, show_steps=False),
}

pd.DataFrame({
    'x found':    [r.x for r in runs.values()],
    'error':      [abs(r.x - 2.25) for r in runs.values()],
    'f(x)':       [r.f for r in runs.values()],
    'iterations': [r.iterations for r in runs.values()],
    'converged':  [r.converged for r in runs.values()],
}, index=runs.keys())

The gradient-based pair (secant, Newton–Raphson) get there in a handful of
iterations, but they need a starting point in the right basin and will converge
to a *maximum* just as happily as a minimum. The bracketing methods are slower
and cannot fail that way.

`newtonraphson` checks the sign of $f''$ at the solution and says so:

In [ ]:
gradientsearch.newtonraphson(lambda x: -(x - 1)**2, 3.0, tol=1e-8)

### 2.3 The iteration table is always kept

`show_steps` only controls printing. The history is on the result either way,
which makes convergence easy to inspect.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].semilogy(r_golden.steps['Iteration'], r_golden.steps['Error'], 'o-')
axes[0].set(xlabel='Iteration', ylabel='|f(X1) - f(X2)|',
            title='Golden section: error decay')
axes[0].grid(alpha=0.3)

nr = runs['Newton-Raphson'].steps
axes[1].semilogy(nr['Iteration'], nr['Error'].clip(lower=1e-16), 'o-', color='crimson')
axes[1].set(xlabel='Iteration', ylabel='|x_k - x_k+1|',
            title='Newton-Raphson: quadratic convergence')
axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

r_golden.steps.head()

## 3. Multivariable unconstrained optimisation

Two methods: Nelder–Mead (`NelderMead_algo.bas`) and Powell's conjugate
direction method (UserForm15). The test case is Rosenbrock's function,

$$f(x) = (1 - x_1)^2 + 100(x_2 - x_1^2)^2$$

whose minimum sits at $(1, 1)$ at the end of a narrow curved valley — steep
across, nearly flat along.

In [ ]:
rosenbrock = lambda x: (1 - x[0])**2 + 100*(x[1] - x[0]**2)**2

r_nm = directsearch.neldermead(rosenbrock, [-1.2, 1.0], tol=1e-12,
                               max_iter=2000, show_steps=False)
r_pw = directsearch.powell(rosenbrock, [-1.2, 1.0], tol=1e-10,
                           show_steps=False)

print(r_nm, end='\n\n')
print(r_pw)
print(f"\nPowell used {r_pw.extras['nfe']} function evaluations")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, res, name in [(axes[0], r_nm, 'Nelder-Mead'),
                      (axes[1], r_pw, "Powell")]:
    plotting.plot_contours(rosenbrock, (-2, 2), (-1, 3), n=90,
                           path=res.steps[['X_1', 'X_2']].values,
                           log_levels=True, ax=ax,
                           title=f'{name}: {res.iterations} iterations')

plt.tight_layout(); plt.show()

Both reach $(1,1)$, but by visibly different routes. Nelder–Mead crawls along
the valley floor by reflecting and contracting its simplex; Powell builds
conjugate directions and takes long strides once it has them.

Neither uses a derivative, which is the reason the workbook favoured them: an
objective living in a spreadsheet cell has no analytical gradient available.

### 3.1 Box constraints

Bounds are applied by clipping every trial point into the box, matching
`Constrain_Point` in the VBA.

In [ ]:
r_box = directsearch.neldermead(rosenbrock, [0.1, 0.1],
                                bounds=[(0, 0.5), (0, 0.5)],
                                tol=1e-12, max_iter=3000, show_steps=False)
print(r_box)
print('\nThe unconstrained optimum (1, 1) is outside the box,')
print('so the solution sits on the x1 = 0.5 face.')

### 3.2 Maximising

Pass `maximize=True` — the same switch as the form's Max/Min option button.
Internally the objective is negated, but the reported `f` is the true value.

In [ ]:
peak = lambda x: -(x[0] - 1)**2 - (x[1] - 2)**2 + 10
directsearch.neldermead(peak, [0, 0], maximize=True, tol=1e-12, show_steps=False)

## 4. Derivatives and the nature of a stationary point

Central differences throughout, with the same step sizes as the VBA
($\Delta x = 10^{-4}$), so the numbers reproduce the spreadsheet rather than
merely approximating it.

In [ ]:
f = lambda x: x[0]**2 + 3*x[0]*x[1] + 2*x[1]**2
grad = calculus.gradient(f, [1.0, 2.0])

In [ ]:
hess = calculus.hessian(f, [1.0, 2.0])
print('\nEigenvalues:', np.round(hess.extras['eigenvalues'], 6))

The eigenvalues have mixed signs, so this Hessian is indefinite and the
stationary point is a saddle. The classification is done for you:

In [ ]:
cases = {
    'bowl  x1^2 + x2^2':        (lambda x: x[0]**2 + x[1]**2,   [0.0, 0.0]),
    'dome  -x1^2 - x2^2':       (lambda x: -x[0]**2 - x[1]**2,  [0.0, 0.0]),
    'saddle x1^2 - x2^2':       (lambda x: x[0]**2 - x[1]**2,   [0.0, 0.0]),
}
for name, (fn, pt) in cases.items():
    print(f'{name:26s} -> {calculus.hessian(fn, pt, show_steps=False).extras["nature"]}')

The Jacobian of a system of equations uses the adaptive step from `JACOBI1`,
$\Delta x = 10^{-5}|x_j| + 10^{-5}$:

In [ ]:
system = [lambda x: x[0]**2 + x[1],
          lambda x: x[0] - x[1]**3]
calculus.jacobian(system, [2.0, 3.0])
print('\nExact: [[4, 1], [1, -27]]')

## 5. Matrix operations

Written out rather than delegated to `numpy.linalg`. Gauss elimination uses
maximum column pivoting plus one error-correction pass — the system is solved
twice, the second time on the residual $b - Ax$, and the results summed. That
single refinement step recovers much of the accuracy lost to round-off.

In [ ]:
A = np.array([[4.0, -2.0,  1.0],
              [-2.0, 4.0, -2.0],
              [1.0, -2.0,  4.0]])
b = np.array([11.0, -16.0, 17.0])

x = matrixops.gauss_pivot(A, b)

In [ ]:
Ainv = matrixops.inverse(A, show_steps=False)
print('A @ inv(A) = I :', np.allclose(Ainv @ A, np.eye(3)))
print('determinant    :', matrixops.determinant(A))

### Eigenvalues without an eigenvalue routine

`EigenValues_SUB` took an unusual approach, reproduced here: sample
$\det(A - \lambda I)$ at $n+1$ values of $\lambda$, solve the resulting
Vandermonde system to recover the characteristic polynomial, then factor that
polynomial by Bairstow's method (repeated quadratic deflation).

The sample points were random in the VBA, so results wobbled between runs;
here a `seed` argument makes it reproducible.

In [ ]:
got  = np.sort(matrixops.eigenvalues(A, show_steps=False))
want = np.sort(np.linalg.eigvals(A).real)

pd.DataFrame({'comxl (Bairstow)': got,
              'numpy.linalg': want,
              'difference': np.abs(got - want)})

## 6. Linear programming

The simplex tableau method from `LinearProgramming.bas`. The problem matrix
follows the workbook's layout — one row per constraint plus a final objective
row, one column per variable plus a final right-hand-side column:

$$\begin{array}{c|ccc}
 & x_1 & x_2 & \text{RHS} \\ \hline
c_1 & 2 & 1 & 10 \\
c_2 & 1 & 3 & 15 \\
z   & 4 & 3 & 0
\end{array}
\qquad
\begin{aligned}
\max\ & z = 4x_1 + 3x_2 \\
\text{s.t. } & 2x_1 + x_2 \le 10 \\
& x_1 + 3x_2 \le 15
\end{aligned}$$

In [ ]:
problem = [[2, 1, 10],
           [1, 3, 15],
           [4, 3,  0]]

lp = linprog.simplex(problem, mode='max')

Every tableau is printed, exactly as the form wrote them down the worksheet.
Watch the objective row: pivoting continues while it holds a negative entry,
and stops when none remains.

### 6.1 Corner points of the feasible region

`feasible_vertices` intersects every pair of constraint boundaries and keeps
the intersections that satisfy all the other constraints. Those survivors are
the corners of the feasible polygon — and by the fundamental theorem of linear
programming, the only places an optimum can sit.

Non-negativity ($x_1, x_2 \ge 0$) is imposed by default, so the axes count as
boundaries too and the origin appears as a corner.

In [ ]:
corners = linprog.feasible_vertices(a=[[2, 1], [1, 3]], b=[10, 15],
                                    senses=['<=', '<='])

The `lines` column records which two boundaries met at each corner, which is
how you read off *why* a vertex exists. The simplex optimum must be one of
these — and it is: $(3, 4)$, where C1 and C2 cross.

In [ ]:
print('simplex optimum :', lp.x)
print('is a corner     :', any(np.allclose([r.x1, r.x2], lp.x, atol=1e-8)
                               for r in corners.itertuples()))

### 6.2 The feasible region, shaded

The region is filled and cross-hatched with a grid, its corners taken from the
exact intersection calculation above, so the edges are crisp rather than the
ragged outline a sampled grid would give. Constraint boundaries are dotted,
following the original workbook chart, each with a fainter companion line
offset toward its feasible side to show which way the inequality points.

Every corner is marked and labelled with its coordinates in $(x, y)$ form.
Labels are boxed so they stay legible over the hatching, and are nudged inward
for corners sitting on an axis so nothing is written off the edge of the
figure.

In [ ]:
ax = plotting.plot_constraints_2d(
    a=[[2, 1], [1, 3]], b=[10, 15], senses=['<=', '<='],
    xlim=(0, 8), ylim=(0, 8),
    point=tuple(lp.x), title='Feasible region, corner points and optimum')
plt.show()

print(f'Optimum at x1 = {lp.x[0]:g}, x2 = {lp.x[1]:g}, z = {lp.f:g}')

The `hatch` argument takes any matplotlib pattern — `'++'` for a grid (the
default), `'xx'` for a diagonal mesh, `'//'` for diagonal lines, or `None` for
a plain fill.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(21, 4.8))
for axis, pattern, name in zip(axes, ['++', 'xx', '//', None],
                               ["grid '++'", "mesh 'xx'", "diagonal '//'",
                                'plain (None)']):
    plotting.plot_constraints_2d([[2, 1], [1, 3]], [10, 15], ['<=', '<='],
                                 xlim=(0, 6), ylim=(0, 6), hatch=pattern,
                                 ax=axis, title=name)
plt.tight_layout(); plt.show()

Because the optimum always sits at a corner, you can solve a small LP by
evaluating the objective at every vertex — the graphical method, and a useful
check on the simplex result:

In [ ]:
corners = corners.assign(z=lambda d: 4*d.x1 + 3*d.x2)
print(corners.to_string(index=False))
best = corners.loc[corners.z.idxmax()]
print('Best corner: ({:g}, {:g}) with z = {:g}'.format(best.x1, best.x2, best.z))
print(f'Simplex agrees: {np.isclose(best.z, lp.f)}')

### 6.3 Awkward regions

Three cases worth seeing: mixed senses, an unbounded region clipped to the
viewing box so it can still be shaded, and an equality constraint that
collapses the region to a line segment with no area at all.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))

plotting.plot_constraints_2d([[1, 1], [2, 1], [1, 3]], [2, 10, 15],
                             ['>=', '<=', '<='], xlim=(0, 8), ylim=(0, 8),
                             ax=axes[0], title='Mixed >= and <=')

plotting.plot_constraints_2d([[1, 1], [1, -1]], [3, 1], ['>=', '<='],
                             xlim=(0, 8), ylim=(0, 8), ax=axes[1],
                             title='Unbounded (clipped to the box)')

plotting.plot_constraints_2d([[1, 1], [2, 1]], [4, 10], ['=', '<='],
                             xlim=(0, 8), ylim=(0, 8), ax=axes[2],
                             title='Equality: region is a segment')

plt.tight_layout(); plt.show()

In [ ]:
for label, args in [
    ('mixed senses', ([[1, 1], [2, 1], [1, 3]], [2, 10, 15], ['>=', '<=', '<='])),
    ('unbounded',    ([[1, 1], [1, -1]], [3, 1], ['>=', '<='])),
    ('equality',     ([[1, 1], [2, 1]], [4, 10], ['=', '<='])),
    ('infeasible',   ([[1, 1], [1, 1]], [2, 8], ['<=', '>='])),
]:
    v = linprog.feasible_vertices(*args, show_steps=False)
    pts = [(round(r.x1, 3), round(r.x2, 3)) for r in v.itertuples()]
    print(f'{label:14s} {len(v)} vertices  {pts}')

An unbounded region has only the two *true* corners `feasible_vertices`
reports; the extra points needed to close the shaded polygon come from the box
edges and are kept separate, in `linprog.region_polygon`. The infeasible case
returns an empty frame rather than raising.

### Minimisation by duality

The workbook solved a minimisation by transposing the problem matrix and
solving the dual, which is why the primal minimum is read out of the objective
row's slack columns rather than the usual place.

In [ ]:
lp_min = linprog.simplex([[1, 2, 4],
                          [3, 1, 6],
                          [6, 3, 0]], mode='min', show_steps=False)
print(lp_min)
print(f'\nCheck: 6({lp_min.x[0]:g}) + 3({lp_min.x[1]:g}) = '
      f'{6*lp_min.x[0] + 3*lp_min.x[1]:g}')

## 7. Constrained nonlinear optimisation

### 7.1 Nonlinear systems

`quasinewton` underpins everything in this section. It is Newton's method with
a finite-difference Jacobian, Gauss elimination for the linear solve, and
reduced steps — if a full step does not lower the residual norm, the step is
halved, up to six times. That damping is what makes it usable from a mediocre
starting guess.

Solve $x_1^2 + x_2^2 = 4$ and $x_1 = x_2$, whose positive root is
$(\sqrt 2, \sqrt 2)$:

In [ ]:
constrained.quasinewton([lambda x: x[0]**2 + x[1]**2 - 4,
                         lambda x: x[0] - x[1]],
                        [1.0, 0.5])
print('\nExact:', np.round([np.sqrt(2), np.sqrt(2)], 6))

### 7.2 Sequential quadratic programming

Build the Lagrangian $L(x, \lambda) = f(x) + \sum_j \lambda_j g_j(x)$ and take
Newton steps on its stationarity conditions:

$$z \leftarrow z - H(L)^{-1} \nabla L, \qquad z = [x_1 \ldots x_n, \lambda_1 \ldots \lambda_m]$$

Minimise $x_1^2 + x_2^2$ subject to $x_1 + x_2 = 4$. The answer is $(2,2)$ with
multiplier $\lambda = -4$.

In [ ]:
objective = lambda x: x[0]**2 + x[1]**2
L = constrained.lagrangian(objective, [lambda x: x[0] + x[1] - 4], n_vars=2)

sqp = constrained.sqp(L, x0=[0.5, 0.5, 0.1], objective=objective, n_vars=2)

In [ ]:
print('variables   :', np.round(sqp.extras['variables'], 8))
print('multipliers :', np.round(sqp.extras['multipliers'], 8))

A harder one: the nearest point on the unit circle to $(1, 2)$, which is
$(1,2)/\sqrt 5$.

In [ ]:
f2 = lambda x: (x[0] - 1)**2 + (x[1] - 2)**2
L2 = constrained.lagrangian(f2, [lambda x: x[0]**2 + x[1]**2 - 1], n_vars=2)

sqp2 = constrained.sqp(L2, [0.5, 0.5, 0.5], objective=f2, n_vars=2,
                       show_steps=False)
print(sqp2)
print('\nExact:', np.round(np.array([1, 2]) / np.sqrt(5), 8))

This SQP takes full Newton steps: no line search, no BFGS update, no active-set
logic. That is what the original did, and it means fast convergence near a
solution but sensitivity to the starting guess.

### 7.3 Karush–Kuhn–Tucker conditions

For

$$\min f(x) \quad \text{s.t.}\quad g_j(x) \le 0,\ h_i(x) = 0$$

the KKT conditions are

$$\nabla f + \sum_j \mu_j \nabla g_j + \sum_i \lambda_i \nabla h_i = 0, \qquad
\mu_j g_j = 0, \quad \mu_j \ge 0, \quad g_j \le 0$$

The complementary-slackness products $\mu_j g_j = 0$ make this combinatorial:
each inequality is either *active* ($g_j = 0$, $\mu_j$ free) or *inactive*
($\mu_j = 0$). `kkt` enumerates all $2^m$ cases, solves each with
`quasinewton`, and keeps whichever satisfy primal and dual feasibility — the
same case analysis UserForm16 did through its branching.

Minimise $(x_1-2)^2 + (x_2-1)^2$ subject to $x_1 + x_2 \le 2$, $x_1, x_2 \ge 0$.
The unconstrained optimum $(2,1)$ violates the first constraint.

In [ ]:
kkt = constrained.kkt(
    lambda x: (x[0] - 2)**2 + (x[1] - 1)**2, n_vars=2,
    inequality=[lambda x: x[0] + x[1] - 2,      # g1
                lambda x: -x[0],                # g2:  x1 >= 0
                lambda x: -x[1]],               # g3:  x2 >= 0
    x0=[0.5, 0.5])

Read the case table: the interior solution is infeasible, several cases give a
negative multiplier (violating $\mu \ge 0$), and only the case with $g_1$
active survives — giving $(1.5, 0.5)$, the projection of $(2,1)$ onto the
constraint line.

Enumeration is exponential in the number of inequalities. That is fine for the
teaching problems the workbook targeted; past roughly ten inequalities you want
an active-set or interior-point method instead.

In [ ]:
ax = plotting.plot_constraints_2d(
    a=[[1, 1], [-1, 0], [0, -1]], b=[2, 0, 0],
    senses=['<=', '<=', '<='], xlim=(-0.5, 3), ylim=(-0.5, 3),
    point=tuple(kkt.x), title='KKT solution')

th = np.linspace(0, 2*np.pi, 200)
for radius in [0.3, 0.707, 1.2]:
    ax.plot(2 + radius*np.cos(th), 1 + radius*np.sin(th),
            color='grey', lw=0.8, ls='--', alpha=0.7)
ax.scatter([2], [1], marker='x', s=110, color='black')
ax.annotate('unconstrained\noptimum', (2, 1), textcoords='offset points',
            xytext=(10, -28), fontsize=9)
plt.show()

The dashed circles are contours of the objective. The solution is where the
lowest reachable contour touches the constraint line — tangency, which is
exactly what the stationarity condition encodes.

## 8. Saving figures

Every plotting function takes the same four arguments for output. `save=True`
writes the figure under an automatic name; passing a string uses that name
instead, with the extension choosing the format. `directory` defaults to the
current working directory and is created if it does not exist. `show=True`
calls `plt.show()` before returning.

In [ ]:
import tempfile, os
outdir = tempfile.mkdtemp()

plotting.plot_constraints_2d([[2, 1], [1, 3]], [10, 15], ['<=', '<='],
                             xlim=(0, 8), ylim=(0, 8),
                             save=True, directory=outdir)

plotting.plot_function(quartic, 0, 4, save='quartic.pdf', directory=outdir)

plotting.plot_contours(rosenbrock, (-2, 2), (-1, 3), n=40, log_levels=True,
                       save='rosenbrock.svg', directory=outdir)

print()
print('written:', sorted(os.listdir(outdir)))

`save_figure` is also available directly, for a figure you assembled yourself:

```python
fig, ax = plt.subplots()
...
plotting.save_figure(fig, 'my_chart.png', directory='figures', dpi=300)
```

It accepts an Axes as well as a Figure, appends `.png` when the name has no
extension, and returns the absolute path it wrote to.

## 9. What every solver returns

A single `Result` object, whatever the method:

| Attribute | Meaning |
|---|---|
| `.x` | the solution (scalar or array) |
| `.f` | objective value there |
| `.iterations` | iterations performed |
| `.converged` | whether the tolerance was met before the cap |
| `.message` | termination reason in words |
| `.steps` | full iteration history as a DataFrame |
| `.method` | which method produced it |
| `.extras` | method-specific extras — gradient, Hessian, tableaux, multipliers |
| `.plot_convergence()` | quick error-vs-iteration plot |
| `.display()` | print the table then the summary |

In [ ]:
r = gradientsearch.goldensection(quartic, 0, 4, tol=1e-8, show_steps=False)

print(r, '\n')
print('steps shape :', r.steps.shape)
print('extras keys :', list(directsearch.powell(rosenbrock, [-1.2, 1.0],
                                                show_steps=False).extras))
r.steps.tail(3)

## 10. Notes on the port

**Deliberately unchanged.** The termination tests, step sizes, and iteration
caps are the VBA's, so output matches the spreadsheet: golden section stops on
a function-value difference rather than interval width; derivative steps are
$10^{-4}$; Newton–Raphson allows 20 iterations; the simplex allows 40 pivots.

**Fixes to genuine bugs**, each documented in the source:

- `neldermead`'s constrained starting simplex used a fixed 0.1 perturbation
  regardless of box width, producing a degenerate simplex and premature
  convergence. It now scales with the box.
- `gauss_pivot` checked pivots $1 \ldots n-1$ but not the last one, dividing by
  zero on rank-deficient systems — which arise routinely inside the KKT
  enumeration.
- The bounds parser could not distinguish `[(lo, hi), ...]` from
  `((lo...), (hi...))` for two variables, silently transposing the box. It now
  disambiguates using the requirement that a lower bound cannot exceed its
  upper bound.

- `neldermead` judged convergence purely by the spread of function values.
  Vertices placed symmetrically about the minimum have equal values, so that
  test collapses to zero while the simplex is still wide: on $(x-3)^2$ from
  $x_0 = 0$ the original stops at 2.9. A second test on the simplex's
  geometric size (`xtol`) is now required as well.
- `gradient` and `hessian` always built a `Result` with a pandas DataFrame,
  even when called from inside a solver that throws everything away but the
  numbers — about 70% of `kkt`'s runtime. The solvers now call
  `gradient_vector` and `hessian_matrix`, which return plain arrays: identical
  values, 12x faster.

**Testing.** 104 tests in `tests/test_comxl.py`, each checking a solver against a
problem with a known analytical answer, plus the docstring examples:

```bash
pytest tests/ -q
pytest --doctest-modules comxl/ -q
```